# Recurrent Neural Network on Alcohol Sales Dataset
This notebook loads an alcohol sales time series dataset, preprocesses it, builds an LSTM recurrent neural network, trains it, and forecasts future sales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from math import sqrt

In [ ]:
# Load dataset
# Replace with your file path if needed
df = pd.read_csv('Alcohol_Sales.csv')
df.head()

In [ ]:
# Basic preprocessing
# Assumes first column is date and second column is sales
df.iloc[:,0] = pd.to_datetime(df.iloc[:,0])
df = df.sort_values(df.columns[0])
df.set_index(df.columns[0], inplace=True)
series = df.iloc[:,0].astype(float)
series.plot(figsize=(12,4), title='Alcohol Sales')
plt.show()

In [ ]:
# Scale data
scaler = MinMaxScaler()
values = scaler.fit_transform(series.values.reshape(-1,1))

In [ ]:
# Create sequences
def create_sequences(data, look_back=12):
    X, y = [], []
    for i in range(len(data)-look_back):
        X.append(data[i:i+look_back, 0])
        y.append(data[i+look_back, 0])
    return np.array(X), np.array(y)

look_back = 12
X, y = create_sequences(values, look_back)
X = X.reshape((X.shape[0], X.shape[1], 1))
X.shape, y.shape

In [ ]:
# Train-test split
split = int(len(X)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
# Build LSTM model
model = Sequential([
    LSTM(64, activation='tanh', input_shape=(look_back,1)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Train
history = model.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test,y_test), verbose=1)

In [ ]:
# Plot loss
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Training Loss')
plt.show()

In [ ]:
# Predict
pred = model.predict(X_test)
pred_inv = scaler.inverse_transform(pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1,1))

rmse = sqrt(mean_squared_error(y_test_inv, pred_inv))
mae = mean_absolute_error(y_test_inv, pred_inv)
print('RMSE:', rmse)
print('MAE:', mae)

In [ ]:
# Visualize predictions
plt.figure(figsize=(12,4))
plt.plot(y_test_inv, label='Actual')
plt.plot(pred_inv, label='Predicted')
plt.legend()
plt.title('Alcohol Sales Forecast')
plt.show()

In [ ]:
# Forecast next 12 periods
last_seq = values[-look_back:].reshape(1,look_back,1)
future = []
curr = last_seq.copy()
for _ in range(12):
    nxt = model.predict(curr, verbose=0)[0,0]
    future.append(nxt)
    curr = np.append(curr[:,1:,:], [[[nxt]]], axis=1)

future_sales = scaler.inverse_transform(np.array(future).reshape(-1,1))
print(future_sales.flatten())